# Subtask 1 – Maximum Entropy Model

This notebook trains a Maximum Entropy model for Subtask 1 and generates the prediction file required for external evaluation.

## 1. Imports

In [ ]:
# Standard
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from src.utils.seed import set_seed
from src.subtask1.models.maxent import MaxEnt
from src.subtask1.models.autoencoder import BinaryAutoencoder

In [ ]:
set_seed(123)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. Data Loading

In [ ]:
# Training data
df_train = pd.read_csv("data/subtask1/train_data_enriched_1311.csv")
df_train = df_train.dropna(subset=["vector_10_binary_llm", "valence", "arousal"])

# Test data
df_test_template = pd.read_csv("data/subtask1/subtask1-template.csv")
df_test_features = pd.read_csv("data/subtask1/test_data_enriched_1701.csv")

df_test = df_test_template.merge(
    df_test_features,
    on=["user_id", "text_id"],
    how="left"
)

# Gold labels (evaluation only)
df_test_labels = pd.read_csv("data/subtask1/test_labels_subtask1.csv")


In [ ]:
train_users = set(df_train["user_id"].unique())
test_users = set(df_test["user_id"].unique())

seen_users = test_users.intersection(train_users)
unseen_users = test_users.difference(train_users)

## 3. Feature Builders

In [ ]:
NUM_LEVELS = 6
INPUT_DIM = 10 * NUM_LEVELS

def parse_and_one_hot(vec_str):
    vals = [int(v) for v in vec_str.split(",")]
    out = np.zeros(INPUT_DIM, dtype=np.float32)

    for i, v in enumerate(vals):
        out[i * NUM_LEVELS + v] = 1.0

    return out

def build_X(df):
    return np.stack(df["vector_10_binary_llm"].apply(parse_and_one_hot))

def build_y(df):
    y_val = df["valence"].map({-2:0, -1:1, 0:2, 1:3, 2:4}).values
    y_aro = df["arousal"].values
    return y_val, y_aro

## 4. Split training users into train/val

In [ ]:
# unique users from TRAIN SET ONLY
users = df_train["user_id"].unique()

train_users, val_users = train_test_split(
    users,
    test_size=0.2,
    random_state=42
)

df_train_final = df_train[df_train["user_id"].isin(train_users)]
df_val = df_train[df_train["user_id"].isin(val_users)]

## 5. Tensor Construction

In [ ]:
# =========================
# TRAIN
# =========================
X_train = torch.tensor(
    build_X(df_train_final),
    dtype=torch.float32,
    device=device
)

y_val_train, y_aro_train = build_y(df_train_final)

y_val_train = torch.tensor(y_val_train, dtype=torch.long, device=device)
y_aro_train = torch.tensor(y_aro_train, dtype=torch.long, device=device)

# =========================
# VALIDATION
# =========================
X_val = torch.tensor(
    build_X(df_val),
    dtype=torch.float32,
    device=device
)

y_val_val, y_aro_val = build_y(df_val)

y_val_val = torch.tensor(y_val_val, dtype=torch.long, device=device)
y_aro_val = torch.tensor(y_aro_val, dtype=torch.long, device=device)

# =========================
# TEST (NO LABELS)
# =========================
X_test = torch.tensor(
    build_X(df_test),
    dtype=torch.float32,
    device=device
)

## 6. Autoencoder

In [ ]:
X_all = torch.cat(
    [X_train, X_val, X_test],
    dim=0
)

In [ ]:
ae = BinaryAutoencoder(input_dim=60, latent_dim=10).to(device)

opt = torch.optim.Adam(ae.parameters(), lr=1e-3)

loss_fn = nn.BCELoss()

for epoch in range(5000):
    opt.zero_grad()
    x_hat, _ = ae(X_all)
    loss = loss_fn(x_hat, X_all)
    loss.backward()
    opt.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | AE loss: {loss.item():.4f}")

## 7. Encode Latent States

In [ ]:
@torch.no_grad()
def encode_latent(ae, X):
    _, z = ae(X)
    return z.round().cpu().numpy().astype(np.float32)

Z_train = encode_latent(ae, X_train)
Z_val   = encode_latent(ae, X_val)

## 8. Build MaxEnt Training Matrix

In [ ]:
def one_hot_valence(v):
    v = int(v) 
    vec = np.zeros(5, dtype=np.float32)
    vec[v + 2] = 1.0
    return vec


def one_hot_arousal(a):
    a = int(a)
    vec = np.zeros(3, dtype=np.float32)
    vec[a] = 1.0
    return vec

In [ ]:
X_maxent = []

for i in range(len(df_train_final)):
    z = Z_train[i]
    v = one_hot_valence(df_train_final.iloc[i]["valence"])
    a = one_hot_arousal(df_train_final.iloc[i]["arousal"])
    X_maxent.append(np.concatenate([z, v, a]))

X_maxent = np.array(X_maxent, dtype=np.float32)

## 9. Train MaxEnt

In [ ]:
STATE_DIM = 10 + 5 + 3  # = 18

maxent = MaxEnt(n=STATE_DIM, device=device)

maxent.fit(
    X_maxent,
    lr=5e-2,
    steps=6000,
    patience=400,
    lambda_=1e-4,
    verbose=True
)

## 10. Inference

In [ ]:
@torch.no_grad()
def predict_maxent_with_probs(maxent, ae, X, tau=1.0):
    """
    Returns:
      val_probs: (N, 5)
      aro_probs: (N, 3)
    """
    device = maxent.device
    X = X.to(device)
    states = maxent.states.to(device)

    # ---- latent probabilities ----
    z_prob = ae.get_z_prob(X)        # (N, 10)
    z = (z_prob > 0.5).float()        # hard latents

    val_probs_all = []
    aro_probs_all = []

    for zi in z:
        # Clamp latent variables
        mask = (states[:, :10] == zi).all(dim=1)
        compatible = states[mask]

        energies = maxent._energy(compatible)
        probs = torch.exp(-energies / tau)
        probs = probs / probs.sum()

        # ----- valence marginal -----
        val_probs = []
        for k in range(5):
            mask_v = compatible[:, 10 + k] == 1
            val_probs.append(probs[mask_v].sum())
        val_probs = torch.stack(val_probs)

        # ----- arousal marginal -----
        aro_probs = []
        for k in range(3):
            mask_a = compatible[:, 15 + k] == 1
            aro_probs.append(probs[mask_a].sum())
        aro_probs = torch.stack(aro_probs)

        val_probs_all.append(val_probs.cpu())
        aro_probs_all.append(aro_probs.cpu())

    return torch.stack(val_probs_all), torch.stack(aro_probs_all)

In [ ]:
@torch.no_grad()
def inference_subtask1(maxent, ae, X_test, df_test):

    maxent.eval()
    ae.eval()

    # Get probability distributions
    val_probs, aro_probs = predict_maxent_with_probs(
        maxent,
        ae,
        X_test
    )

    # Expectation decoding
    device = val_probs.device

    val_grid = torch.tensor([-2, -1, 0, 1, 2],
                            device=device,
                            dtype=torch.float32)

    aro_grid = torch.tensor([0, 1, 2],
                            device=device,
                            dtype=torch.float32)

    val_expected = (val_probs * val_grid).sum(dim=1)
    aro_expected = (aro_probs * aro_grid).sum(dim=1)

    # Build submission
    submission = pd.DataFrame({
        "user_id": df_test["user_id"].values,
        "text_id": df_test["text_id"].values,
        "pred_valence": val_expected.cpu().numpy(),
        "pred_arousal": aro_expected.cpu().numpy()
    })

    return submission

In [ ]:
submission = inference_subtask1(maxent, ae, X_test, df_test)

submission.to_csv("results/subtask1/subtask1_predictions1.csv", index=False)